# Stratified K-fold cross-validation with rule-level targeting metrics

How reliable is action-rule deployment when measured *out of sample*? This
notebook runs `ActionRules.cross_validate()` on each benchmark dataset and
reports three complementary views (Hastie, Tibshirani & Friedman, *Elements
of Statistical Learning*, Ch. 7; Bates, Hastie & Tibshirani 2021):

1. **In-sample (apparent).** Mine + score on the full dataset; an optimistic
   upper bound by construction.
2. **Per-fold range.** Min / median / max across the 5 train/test holdouts —
   captures rule-discovery instability and OOF noise.
3. **CV aggregate.** Across-fold mean with the cluster-by-fold bootstrap 95 %
   CI (the inferential interval recommended by Bates et al.).

Four **targeting strategies** are evaluated:

- `point` — rank by train-fold uplift point estimate.
- `lower` — rank by lower 95 % CI bound (conservative).
- `lower_positive` — further filter to rules whose lower bound is positive.
- `risk_adjusted` — rank by `point - 1.96 · SE`.

Four **metrics** are computed: `uplift_at_k`, `qini`, `auuc`, `profit_at_k`.

**Outputs** (under `notebooks/inference_studies/results/`):
- `cv_multiview.csv` — long form, one row per (dataset × strategy × metric ×
  view).
- `cv_results.csv` — CV aggregate only; matches the historic article export.
- `cv_rule_records.csv` — per-rule per-fold records (rule x fold).

Runtime: ~3-5 minutes total across the four datasets.


In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the repository root from the notebook's working directory.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Repository root with pyproject.toml not found.')

# Import the shared dataset loader (canonical preprocessing for the four
# benchmark datasets used throughout this folder).
sys.path.insert(0, str(REPO_ROOT / 'notebooks' / 'article'))
from _datasets import load_all, load_telco, load_bank_marketing, load_employee_attrition, load_credit_card_default  # noqa: E402

# Outputs for this notebook (kept under the notebook folder so the user can
# inspect them locally without touching the article's canonical CSVs).
RESULTS_DIR = REPO_ROOT / 'notebooks' / 'inference_studies' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
print(f'Repo root:   {REPO_ROOT}')
print(f'Results dir: {RESULTS_DIR.relative_to(REPO_ROOT)}')


In [ ]:
import time
from action_rules import ActionRules

OUT_CSV = RESULTS_DIR / 'cv_results.csv'
OUT_RULES = RESULTS_DIR / 'cv_rule_records.csv'
OUT_MULTIVIEW = RESULTS_DIR / 'cv_multiview.csv'


In [ ]:
N_SPLITS = 5              # smoke: 3
N_BOOTSTRAP = 500         # smoke: 120
N_BOOTSTRAP_OOF = 1000    # smoke: 200
K_FRACTION = 0.20
SEED = 42
# 'cluster_fold' resamples rule records within each fold, computes per-fold
# metrics, and averages across folds — the CI then estimates the same
# fold-mean quantity as the `mean` column.
BOOTSTRAP_DESIGN = 'cluster_fold'


## Run CV on every dataset


In [ ]:
summary_frames, rule_frames, multiview_frames = [], [], []
for cfg in load_all():
    print(f'=== {cfg.name} ===')
    ar = ActionRules(
        min_stable_attributes=cfg.min_stable_attributes,
        min_flexible_attributes=cfg.min_flexible_attributes,
        min_undesired_support=cfg.min_undesired_support,
        min_desired_support=cfg.min_desired_support,
        min_undesired_confidence=cfg.min_undesired_confidence,
        min_desired_confidence=cfg.min_desired_confidence,
        intrinsic_utility_table=cfg.intrinsic_utility_table,
        transition_utility_table=cfg.transition_utility_table,
    )
    t0 = time.perf_counter()
    result = ar.cross_validate(
        cfg.df,
        stable_attributes=cfg.stable_attributes,
        flexible_attributes=cfg.flexible_attributes,
        target=cfg.target,
        target_undesired_state=cfg.target_undesired_state,
        target_desired_state=cfg.target_desired_state,
        n_splits=N_SPLITS,
        n_bootstrap=N_BOOTSTRAP,
        n_bootstrap_oof=N_BOOTSTRAP_OOF,
        bootstrap_design=BOOTSTRAP_DESIGN,
        k_fraction=K_FRACTION,
        random_state=SEED,
        use_sparse_matrix=cfg.use_sparse_matrix,
        compute_insample_baseline=True,
    )
    elapsed = time.perf_counter() - t0
    summary = result.strategy_summary.copy()
    summary.insert(0, 'dataset', cfg.name)
    summary_frames.append(summary)
    rules = result.rule_records.copy()
    rules.insert(0, 'dataset', cfg.name)
    rule_frames.append(rules)

    # Build the three-view long form.
    cv_view = summary.copy()
    cv_view['view'] = 'cv_aggregate'
    cv_view['range_min'] = pd.NA
    cv_view['range_max'] = pd.NA
    cv_view['range_median'] = pd.NA

    fr = (
        result.fold_summary.groupby(['strategy', 'metric', 'metric_target'])['value']
        .agg(['min', 'max', 'median'])
        .reset_index()
        .rename(columns={'min': 'range_min', 'max': 'range_max', 'median': 'range_median'})
    )
    fr.insert(0, 'dataset', cfg.name)
    fr['view'] = 'fold_range'
    fr['mean'] = pd.NA
    fr['std'] = pd.NA
    fr['n_folds'] = result.n_splits
    fr['ci_lower'] = pd.NA
    fr['ci_upper'] = pd.NA

    ins = result.insample_summary.copy() if result.insample_summary is not None else pd.DataFrame()
    if not ins.empty:
        ins = ins.rename(columns={'value': 'mean'})
        ins.insert(0, 'dataset', cfg.name)
        ins['view'] = 'in_sample'
        ins['std'] = pd.NA
        ins['n_folds'] = pd.NA
        ins['ci_lower'] = pd.NA
        ins['ci_upper'] = pd.NA
        ins['range_min'] = pd.NA
        ins['range_max'] = pd.NA
        ins['range_median'] = pd.NA

    multiview_frames.append(pd.concat([ins, fr, cv_view], ignore_index=True, sort=False))
    jaccard = (
        result.rule_stability['jaccard'].mean()
        if result.rule_stability is not None else float('nan')
    )
    print(
        f'  rules/fold = {result.n_rules_per_fold}; '
        f'mean Jaccard overlap = {jaccard:.3f}; elapsed = {elapsed:.1f}s'
    )

summary_df = pd.concat(summary_frames, ignore_index=True)
rule_df = pd.concat(rule_frames, ignore_index=True)
multiview_df = pd.concat(multiview_frames, ignore_index=True, sort=False)
multiview_cols = [
    'dataset', 'strategy', 'metric', 'metric_target', 'view',
    'mean', 'std', 'n_folds', 'ci_lower', 'ci_upper',
    'range_min', 'range_max', 'range_median',
]
multiview_df = multiview_df.reindex(columns=multiview_cols)

summary_df.to_csv(OUT_CSV, index=False)
rule_df.to_csv(OUT_RULES, index=False)
multiview_df.to_csv(OUT_MULTIVIEW, index=False)
print(f'\nwrote {OUT_CSV.relative_to(REPO_ROOT)}')
print(f'wrote {OUT_RULES.relative_to(REPO_ROOT)}')
print(f'wrote {OUT_MULTIVIEW.relative_to(REPO_ROOT)}')


In [ ]:
# Quick view: uplift@20% per dataset x strategy, cross-fold mean
pivot = (
    summary_df[(summary_df['metric'] == 'uplift_at_k') & (summary_df['metric_target'] == 'uplift')]
    .pivot(index='dataset', columns='strategy', values='mean')
    .round(4)
)
print('Uplift@20% (mean across folds), by dataset x strategy:')
print(pivot.to_string())


## Interpretation

- The `cv_aggregate` rows are the recommended **point estimate of out-of-sample
  performance** for each strategy.
- The `in_sample` rows show how much **optimism** the apparent fit carries —
  the gap to the CV aggregate is the *generalization gap*.
- The `fold_range` rows show **stability**: a wide [min, max] window means
  rule discovery is sensitive to the fold split.
- The Jaccard overlap printed per dataset measures how much the *rule set*
  itself fluctuates between folds (1.0 = identical rules, 0.0 = disjoint).
- Cluster-by-fold CIs (`ci_lower`, `ci_upper`) are the inferentially valid
  intervals; the standard `mean +/- 1.96 std / sqrt(K)` interval has below-nominal
  coverage (Bates, Hastie & Tibshirani 2021).
